# LangGraph: Introduzione ai Workflow con LLM

**LangGraph** è una libreria che permette di costruire applicazioni con LLM come **grafi**:
ogni passo è un nodo, e i nodi sono collegati da archi.

In questo notebook introduttivo vedremo 4 pattern fondamentali:

| # | Pattern | Cosa mostra |
|---|---------|-------------|
| 1 | **Nodo singolo** | Come collegare un LLM a LangGraph |
| 2 | **Workflow sequenziale** | Due nodi in serie: l'output del primo diventa input del secondo |
| 3 | **Routing condizionale** | Scegliere il percorso in base al contenuto |
| 4 | **Agente con tools** | Un LLM che usa strumenti in modo autonomo |

```bash
pip install langgraph langchain langchain-openai python-dotenv
```

In [1]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o", temperature=0)
print("Setup completato")

Setup completato


In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

print("Import completati")

Import completati


---

## Concetti Base

Un grafo LangGraph ha tre componenti:

```
    START
      |
   [nodo]   ← funzione Python che legge lo State e restituisce aggiornamenti
      |
     END
```

- **State** – un dizionario (`TypedDict`) con tutti i dati che passano tra i nodi
- **Nodo** – una funzione `(state) -> dict` che legge e aggiorna lo state
- **Arco** – la connessione tra due nodi

### Struttura minima

```python
class State(TypedDict):          # 1. definisci lo State
    input: str
    output: str

def mio_nodo(state: State) -> dict:    # 2. scrivi i nodi
    return {"output": "risultato"}

builder = StateGraph(State)      # 3. costruisci il grafo
builder.add_node("mio_nodo", mio_nodo)
builder.add_edge(START, "mio_nodo")
builder.add_edge("mio_nodo", END)
graph = builder.compile()        # 4. compila

graph.invoke({"input": "ciao"})  # 5. esegui
```

In [3]:
# --- Grafo minimo senza LLM: solo per capire la struttura ---
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class State(TypedDict):
    messaggio: str
    risposta: str

def echo(state: State) -> dict:
    testo = state["messaggio"].upper()
    return {"risposta": f"Ho ricevuto: {testo}"}

builder = StateGraph(State)
builder.add_node("echo", echo)
builder.add_edge(START, "echo")
builder.add_edge("echo", END)
grafo = builder.compile()

risultato = grafo.invoke({"messaggio": "ciao langgraph"})
print(risultato)

{'messaggio': 'ciao langgraph', 'risposta': 'Ho ricevuto: CIAO LANGGRAPH'}


In [4]:
# Visualizza la struttura del grafo
try:
    print(grafo.get_graph().draw_ascii())
except ImportError:
    print("Per visualizzare i grafi, installa: pip install grandalf")

+-----------+  
| __start__ |  
+-----------+  
      *        
      *        
      *        
  +------+     
  | echo |     
  +------+     
      *        
      *        
      *        
 +---------+   
 | __end__ |   
 +---------+   


---

## Pattern 1: Nodo Singolo con LLM

Il caso più semplice: un LLM all'interno di un nodo LangGraph.

```
  domanda
     |
  [rispondi]   ← LLM
     |
  risposta
```

Rispetto a chiamare direttamente `model.invoke(...)`, il grafo ci dà:
- uno **State** esplicito con input e output nominati
- la possibilità di aggiungere altri nodi in futuro senza riscrivere tutto

In [5]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o", temperature=0)

class StateDomanda(TypedDict):
    domanda: str
    risposta: str

def rispondi(state: StateDomanda) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Rispondi in italiano in modo chiaro e conciso: {domanda}"
    )
    result = (prompt | model | StrOutputParser()).invoke({"domanda": state["domanda"]})
    return {"risposta": result}

builder = StateGraph(StateDomanda)
builder.add_node("rispondi", rispondi)
builder.add_edge(START, "rispondi")
builder.add_edge("rispondi", END)
grafo_llm = builder.compile()

# Esecuzione
try:
    result = grafo_llm.invoke({"domanda": "Cos'e' un ETF?"})
    print("Domanda:", result["domanda"])
    print("\nRisposta:", result["risposta"])
except Exception as e:
    print(f"Nota: {type(e).__name__}")
    print("Per eseguire: verifica che OPENAI_API_KEY sia impostato in .env")
    print("Grafo creato correttamente e pronto per l'esecuzione.")

Domanda: Cos'e' un ETF?

Risposta: Un ETF, o "Exchange Traded Fund", è un fondo d'investimento quotato in borsa che replica l'andamento di un indice di riferimento, come l'S&P 500 o il FTSE MIB. Gli ETF combinano le caratteristiche dei fondi comuni di investimento con la flessibilità delle azioni, permettendo agli investitori di acquistare e vendere quote durante l'orario di contrattazione. Sono strumenti popolari per la diversificazione del portafoglio grazie ai loro costi generalmente più bassi e alla trasparenza.


---

## Pattern 2: Workflow Sequenziale

Due (o più) nodi in serie. L'output del primo nodo finisce nello State
e viene letto dal secondo.

```
  articolo
      |
  [riassumi]     → riassunto
      |
  [punti_chiave] → punti_chiave
      |
     END
```

**Quando usarlo:** quando hai una pipeline lineare dove ogni passo dipende dal precedente.

**Vantaggio rispetto a LCEL:** lo State rende esplicito cosa passa tra un passo e l'altro,
e ogni nodo è testabile in isolamento.

In [6]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o", temperature=0)

class StateNotizia(TypedDict):
    articolo: str
    riassunto: str
    punti_chiave: str

def riassumi(state: StateNotizia) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Riassumi questa notizia finanziaria in 2 frasi in italiano:\n\n{articolo}"
    )
    result = (prompt | model | StrOutputParser()).invoke({"articolo": state["articolo"]})
    return {"riassunto": result}

def estrai_punti(state: StateNotizia) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Basandoti su questo riassunto, elenca 3 punti chiave per un investitore:\n\n{riassunto}"
    )
    result = (prompt | model | StrOutputParser()).invoke({"riassunto": state["riassunto"]})
    return {"punti_chiave": result}

builder = StateGraph(StateNotizia)
builder.add_node("riassumi", riassumi)
builder.add_node("estrai_punti", estrai_punti)

builder.add_edge(START, "riassumi")
builder.add_edge("riassumi", "estrai_punti")   # output di riassumi → input di estrai_punti
builder.add_edge("estrai_punti", END)

grafo_seq = builder.compile()
try:
    print(grafo_seq.get_graph().draw_ascii())
except ImportError:
    print("Per visualizzare i grafi, installa: pip install grandalf")

  +-----------+  
  | __start__ |  
  +-----------+  
        *        
        *        
        *        
  +----------+   
  | riassumi |   
  +----------+   
        *        
        *        
        *        
+--------------+ 
| estrai_punti | 
+--------------+ 
        *        
        *        
        *        
  +---------+    
  | __end__ |    
  +---------+    


In [7]:
articolo = (
    "Nvidia shares surged 5% today as the company reported record quarterly revenue "
    "driven by unprecedented demand for its AI chips. CEO Jensen Huang said demand "
    "continues to outpace supply for the H100 and next-generation Blackwell GPUs. "
    "The results beat analyst expectations by a wide margin."
)

try:
    result = grafo_seq.invoke({"articolo": articolo})
    print("RIASSUNTO")
    print("-" * 40)
    print(result["riassunto"])
    print("\nPUNTI CHIAVE PER L'INVESTITORE")
    print("-" * 40)
    print(result["punti_chiave"])
except Exception as e:
    print(f"Nota: {type(e).__name__}")
    print("Per eseguire: verifica che OPENAI_API_KEY sia impostato in .env")
    print("Grafo sequenziale creato correttamente e pronto per l'esecuzione.")

RIASSUNTO
----------------------------------------
Le azioni di Nvidia sono aumentate del 5% dopo che l'azienda ha riportato un fatturato trimestrale record, trainato dalla domanda senza precedenti per i suoi chip AI. Il CEO Jensen Huang ha dichiarato che la domanda per le GPU H100 e Blackwell di nuova generazione continua a superare l'offerta, con risultati che hanno superato di gran lunga le aspettative degli analisti.

PUNTI CHIAVE PER L'INVESTITORE
----------------------------------------
Sulla base del riassunto fornito, ecco tre punti chiave per un investitore:

1. **Crescita della Domanda di Chip AI**: Nvidia sta beneficiando di una domanda senza precedenti per i suoi chip AI, in particolare le GPU H100 e Blackwell di nuova generazione. Questo indica un forte potenziale di crescita nel settore dell'intelligenza artificiale, che potrebbe continuare a sostenere l'aumento delle azioni dell'azienda.

2. **Performance Finanziaria Solida**: L'azienda ha riportato un fatturato trimestr

---

## Pattern 3: Routing Condizionale

Un nodo classifica l'input e lo manda su percorsi diversi.

```
  domanda
      |
  [classifica]
      |
      +-- "MERCATI"  --> [esperto_mercati]  --> END
      +-- "AZIENDE"  --> [esperto_aziende]  --> END
      +-- "ECONOMIA" --> [esperto_economia] --> END
```

**Quando usarlo:** quando risposte diverse richiedono prompt (o modelli) diversi.

**In LangGraph:** `add_conditional_edges(nodo, funzione_di_routing)` dove la funzione
restituisce il **nome del prossimo nodo** come stringa.

In [8]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o", temperature=0)

class StateDomandaFinanziaria(TypedDict):
    domanda: str
    tipo: str
    risposta: str

# Nodo 1: classifica la domanda
def classifica(state: StateDomandaFinanziaria) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Classifica questa domanda in UNA categoria: MERCATI, AZIENDE, ECONOMIA\n"
        "Rispondi solo con la categoria.\n\nDomanda: {domanda}"
    )
    result = (prompt | model | StrOutputParser()).invoke({"domanda": state["domanda"]})
    return {"tipo": result.strip().upper()}

# Funzione di routing: decide il prossimo nodo in base allo State
def scegli_esperto(state: StateDomandaFinanziaria) -> str:
    if "MERCATI" in state["tipo"]:
        return "esperto_mercati"
    elif "AZIENDE" in state["tipo"]:
        return "esperto_aziende"
    return "esperto_economia"

# Nodi specializzati: ognuno usa un prompt diverso
def esperto_mercati(state: StateDomandaFinanziaria) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Sei un esperto di mercati finanziari. Rispondi in modo tecnico ma chiaro:\n{domanda}"
    )
    return {"risposta": (prompt | model | StrOutputParser()).invoke({"domanda": state["domanda"]})}

def esperto_aziende(state: StateDomandaFinanziaria) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Sei un analista aziendale. Rispondi con focus su strategia e fondamentali:\n{domanda}"
    )
    return {"risposta": (prompt | model | StrOutputParser()).invoke({"domanda": state["domanda"]})}

def esperto_economia(state: StateDomandaFinanziaria) -> dict:
    prompt = ChatPromptTemplate.from_template(
        "Sei un economista. Rispondi con una prospettiva macroeconomica:\n{domanda}"
    )
    return {"risposta": (prompt | model | StrOutputParser()).invoke({"domanda": state["domanda"]})}

# Costruzione del grafo
builder = StateGraph(StateDomandaFinanziaria)
builder.add_node("classifica", classifica)
builder.add_node("esperto_mercati", esperto_mercati)
builder.add_node("esperto_aziende", esperto_aziende)
builder.add_node("esperto_economia", esperto_economia)

builder.add_edge(START, "classifica")
builder.add_conditional_edges("classifica", scegli_esperto)  # routing dinamico
builder.add_edge("esperto_mercati", END)
builder.add_edge("esperto_aziende", END)
builder.add_edge("esperto_economia", END)

grafo_route = builder.compile()
try:
    print(grafo_route.get_graph().draw_ascii())
except ImportError:
    print("Per visualizzare i grafi, installa: pip install grandalf")

+-----------+  
| __start__ |  
+-----------+  
       *       
       *       
       *       
+------------+ 
| classifica | 
+------------+ 
       *       
       *       
       *       
  +---------+  
  | __end__ |  
  +---------+  


In [9]:
domande = [
    "Perche' i tassi di interesse influenzano il prezzo delle azioni?",
    "Come si analizzano i risultati trimestrali di Apple?",
    "Cosa si intende per spread BTP-Bund?",
]

try:
    for d in domande:
        result = grafo_route.invoke({"domanda": d})
        print(f"\nDomanda: {d}")
        print(f"Tipo:    {result['tipo']}")
        print(f"Risposta: {result['risposta'][:250]}...")
        print("-" * 60)
except Exception as e:
    print(f"Nota: {type(e).__name__}")
    print("Per eseguire: verifica che OPENAI_API_KEY sia impostato in .env")
    print("Grafo routing creato correttamente e pronto per l'esecuzione.")


Domanda: Perche' i tassi di interesse influenzano il prezzo delle azioni?
Tipo:    ECONOMIA
Risposta: I tassi di interesse influenzano il prezzo delle azioni attraverso diversi canali macroeconomici:

1. **Costo del capitale**: I tassi di interesse rappresentano il costo del denaro. Quando i tassi aumentano, il costo del capitale per le imprese cresc...
------------------------------------------------------------

Domanda: Come si analizzano i risultati trimestrali di Apple?
Tipo:    AZIENDE
Risposta: Analizzare i risultati trimestrali di Apple richiede un approccio strutturato che consideri sia gli aspetti quantitativi che qualitativi. Ecco una guida su come procedere:

### 1. **Analisi dei Ricavi**
- **Segmentazione dei Prodotti**: Esamina i ric...
------------------------------------------------------------

Domanda: Cosa si intende per spread BTP-Bund?
Tipo:    ECONOMIA
Risposta: Lo spread BTP-Bund è un indicatore finanziario che misura la differenza di rendimento tra i titoli di 

---

## Pattern 4: Agente con Tools

Un **agente** è un LLM che può usare strumenti (tools) in modo autonomo:
ragiona, chiama un tool, osserva il risultato, e ripete finché ha la risposta.

```
   domanda
       |
   [agente]
       |
       +-- ha bisogno di un tool --> [tool] --> [agente]
       |                                    (loop)
       +-- ha la risposta --> END
```

I tools sono semplici **funzioni Python** decorate con `@tool`.
LangGraph offre `create_react_agent` per costruire questo pattern in una riga.

In [10]:
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI

model = ChatOpenAI(model="gpt-4o", temperature=0)

@tool
def calcola(espressione: str) -> str:
    """Calcola un'espressione matematica. Esempi: '100 * 1.05', '50000 / 12', '1.07 ** 5'"""
    try:
        risultato = eval(espressione, {"__builtins__": {}}, {})
        return f"{espressione} = {risultato:.4f}"
    except Exception as e:
        return f"Errore nel calcolo: {e}"

@tool
def converti_valuta(importo: float, da: str, a: str) -> str:
    """
    Converte un importo tra valute (tassi approssimativi).
    Valute supportate: EUR, USD, GBP, JPY
    Esempio: converti_valuta(1000, 'USD', 'EUR')
    """
    tassi_vs_eur = {"EUR": 1.0, "USD": 1.09, "GBP": 0.85, "JPY": 163.0}
    da, a = da.upper(), a.upper()
    if da not in tassi_vs_eur or a not in tassi_vs_eur:
        return f"Valuta non supportata. Usa: {', '.join(tassi_vs_eur)}"
    valore_eur = importo / tassi_vs_eur[da]
    risultato = valore_eur * tassi_vs_eur[a]
    return f"{importo:,.2f} {da} = {risultato:,.2f} {a}"

tools = [calcola, converti_valuta]
agente = create_react_agent(model, tools)

print("Agente pronto con tools:", [t.name for t in tools])

Agente pronto con tools: ['calcola', 'converti_valuta']


C:\Users\l_ace\AppData\Local\Temp\ipykernel_6756\64822432.py:33: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agente = create_react_agent(model, tools)


In [11]:
# L'agente sceglie autonomamente quali tool usare e in quale ordine
domanda = (
    "Ho 10.000 dollari americani. "
    "Quanto valgono in euro? "
    "E se li investissi con un rendimento del 7% annuo per 5 anni, "
    "quanto avrei alla fine sempre in euro?"
)

try:
    result = agente.invoke({"messages": [HumanMessage(content=domanda)]})
    print("Domanda:", domanda)
    print("\nRisposta dell'agente:")
    print(result["messages"][-1].content)
except Exception as e:
    print(f"Nota: {type(e).__name__}")
    print("Per eseguire: verifica che OPENAI_API_KEY sia impostato in .env")
    print("Agente creato correttamente con tools: calcola, converti_valuta")

Domanda: Ho 10.000 dollari americani. Quanto valgono in euro? E se li investissi con un rendimento del 7% annuo per 5 anni, quanto avrei alla fine sempre in euro?

Risposta dell'agente:
I tuoi 10.000 dollari americani valgono circa 9.174,31 euro. Se li investissi con un rendimento del 7% annuo per 5 anni, alla fine avresti circa 12.867,44 euro.


In [12]:
# Secondo esempio: domanda che richiede piu' passaggi
domanda2 = (
    "Un portafoglio ha 3 asset: "
    "5.000 USD in azioni, 2.000 GBP in obbligazioni, 500.000 JPY in liquidita'. "
    "Convertili tutti in EUR e dimmi il totale."
)

try:
    result2 = agente.invoke({"messages": [HumanMessage(content=domanda2)]})
    print("Domanda:", domanda2)
    print("\nRisposta:")
    print(result2["messages"][-1].content)
except Exception as e:
    print(f"Nota: {type(e).__name__}")
    print("Per eseguire: verifica che OPENAI_API_KEY sia impostato in .env")
    print("Agente pronto per l'esecuzione.")

Domanda: Un portafoglio ha 3 asset: 5.000 USD in azioni, 2.000 GBP in obbligazioni, 500.000 JPY in liquidita'. Convertili tutti in EUR e dimmi il totale.

Risposta:
Il totale del portafoglio convertito in EUR è di 10.007,58 EUR.


---

## Riepilogo

| Pattern | Struttura | Quando usarlo |
|---------|-----------|---------------|
| **Nodo singolo** | `START → [llm] → END` | Sostituire una singola chiamata LLM |
| **Sequenziale** | `[nodo_a] → [nodo_b] → [nodo_c]` | Pipeline lineare, ogni passo usa l'output del precedente |
| **Routing** | `[classifica] → [path_a \| path_b]` | Risposte/comportamenti diversi in base all'input |
| **Agente** | `[llm] ↔ [tools]` | Il modello decide autonomamente cosa fare |

### Cosa abbiamo visto

- **`TypedDict`** per definire lo State in modo chiaro
- **`add_edge`** per connessioni sequenziali
- **`add_conditional_edges`** per il routing dinamico
- **`@tool` + `create_react_agent`** per agenti autonomi
- **`graph.get_graph().draw_ascii()`** per visualizzare la struttura

### Prossimi passi

Una volta capiti questi pattern di base, i pattern avanzati di LangGraph aggiungono:
- **Parallelizzazione** – nodi eseguiti in parallelo (fan-out/fan-in)
- **Loop con feedback** – cicli genera → valuta → migliora
- **Map-Reduce** – elaborazione di liste dinamiche di items
- **Memoria e persistenza** – `MemorySaver` per conservare lo State tra sessioni